# IEEE 118-Bus System - Security-Constrained DC Optimal Power Flow (SC-DCOPF)

Branch-oriented SC-DCOPF following the thesis formulation. Minimize expected generation
cost while keeping the dispatch feasible across a set of single-element ($N-1$) contingencies
$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$.

$$\min_{P,f,\theta} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

subject to nodal balance (3.2), generator limits (3.3), thermal limits (3.4), the DC flow
law (3.5)/(3.5b), and the reference-bus constraint (3.6).

All input data is read from `processed/`. Per-unit on a
100 MVA base.

**Build strategy:** we start with the base case only ($\mathcal{C}=\{0\}$) to validate the
mechanics against a plain DCOPF, then switch on $\mathcal{C}_T$ and $\mathcal{C}_G$.

## 1. Imports and processed data

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [2]:
# This notebook lives in a model subfolder (e.g. project/sc-dcopf/), while the
# shared processed data lives one level up. ROOT points to the project data root so paths
# survive folder reorganizations (change here only).
# SOURCE picks which processed dataset to use: 'matpower' or 'scuc'.
ROOT   = ".."
SOURCE = "matpower"   # 'matpower' or 'scuc'
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")

base_mva = 100 MVA | slack bus = 69
generators = 19 | buses = 118 | branches = 186


## 2. Index sets and branch-oriented structures

We build the sets and lookups the formulation needs. The nodal balance (3.2) is written per
bus using the sets of **outgoing** lines $\Omega_i^+$ (branches with `from_bus == i`) and
**incoming** lines $\Omega_i^-$ (`to_bus == i`), plus the generators attached to each bus
$\mathcal{G}_i$.

In [3]:
# --- sets ---
set_N = loads["bus"].tolist()          # buses      (N)
set_G = gens["gen_id"].tolist()        # generators (G)
set_L = branches["branch_id"].tolist() # branches   (L)

# --- per-element parameters ---
par_pd     = dict(zip(loads["bus"],          loads["pd_pu"]))      # P_{d,i}
par_g_bus  = dict(zip(gens["gen_id"],        gens["bus"]))         # bus of generator g
par_pmin   = dict(zip(gens["gen_id"],        gens["pmin_pu"]))     # P_g^min
par_pmax   = dict(zip(gens["gen_id"],        gens["pmax_pu"]))     # P_g^max
par_cost   = dict(zip(gens["gen_id"],        gens["c1"]))          # c_g  ($/MWh)
par_from   = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to     = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x      = dict(zip(branches["branch_id"], branches["x_pu"]))    # x_{ij}
par_rate   = dict(zip(branches["branch_id"], branches["rate_pu"])) # F_{ij}^max

# --- subset G_i : generators connected to each bus ---
set_gens_at = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)

# --- subsets Omega_i^+ (leaving bus i) and Omega_i^- (entering bus i) ---
set_out_lines = {i: [] for i in set_N}   # Omega_i^+  (from_bus == i)
set_in_lines  = {i: [] for i in set_N}   # Omega_i^-  (to_bus == i)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

# --- integrity checks ---
assert SLACK_BUS in set_N, "Slack bus not found among buses."
assert sum(len(v) for v in set_gens_at.values()) == len(set_G)
assert sum(len(v) for v in set_out_lines.values()) == len(set_L)

print(f"|N| = {len(set_N)} buses | |G| = {len(set_G)} generators | |L| = {len(set_L)} branches")
print(f"Bus {SLACK_BUS} (slack): gens={set_gens_at[SLACK_BUS]}, "
      f"lines out={len(set_out_lines[SLACK_BUS])}, lines in={len(set_in_lines[SLACK_BUS])}")

|N| = 118 buses | |G| = 19 generators | |L| = 186 branches
Bus 69 (slack): gens=[12], lines out=3, lines in=3


## 3. Operating states $\mathcal{C}$, availability $N1$, and probabilities $\rho$

$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$: the base case plus one scenario per
single-element contingency. $\mathcal{C}_T$ uses **non-radial lines only** (bridges are
excluded, since their outage would island a bus); $\mathcal{C}_G$ uses the real generators.

In [17]:
INCLUDE_CT = False   # transmission contingencies (non-radial lines)
INCLUDE_CG = False    # generator contingencies

# --- run identity: model / formulation / (config comes from the switches above) ---
MODEL       = "sc-dcopf"
FORMULATION = "redispatch_with_exclusions"  # redispatch_only | redispatch_with_exclusions | with_load_shedding

# The set of contingencies to drop is TIED to the formulation, so choosing the formulation
# automatically sets the right behaviour (no risk of mismatched settings).
# The excluded elements were identified by vulnerability.ipynb (no corrective solution).
if FORMULATION == "redispatch_with_exclusions":
    EXCLUDE = [("line", 7),    # line 8-5
               ("line", 50),   # line 38-37
               ("gen", 0)]     # generator at bus 10 (505 MW)
else:  # 'redispatch_only' (expect infeasibility) or 'with_load_shedding' (handled by shedding)
    EXCLUDE = []

# Eligible contingency elements (drop the excluded ones)
CT_lines = [l for l in branches.loc[~branches["is_bridge"], "branch_id"].tolist()
            if ("line", l) not in EXCLUDE]                 # non-radial, not excluded
CG_gens  = [g for g in set_G if ("gen", g) not in EXCLUDE] # real generators, not excluded

# Build the scenario list. Each scenario knows which element (if any) is out of service.
scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None,
              "desc": "base case (all in service)"}]

if INCLUDE_CT:
    for l in CT_lines:
        scenarios.append({"id": len(scenarios), "type": "line",
                          "out_line": l, "out_gen": None, "desc": f"line {l} out"})
if INCLUDE_CG:
    for g in CG_gens:
        scenarios.append({"id": len(scenarios), "type": "gen",
                          "out_line": None, "out_gen": g, "desc": f"gen {g} out"})

set_C = [s["id"] for s in scenarios]     # C : operating states
scen  = {s["id"]: s for s in scenarios}  # lookup by scenario id

# N1 availability (0 = out of service, 1 = in service)
def line_on(l, c):
    return 0 if scen[c]["out_line"] == l else 1

def gen_on(g, c):
    return 0 if scen[c]["out_gen"] == g else 1

print(f"INCLUDE_CT={INCLUDE_CT} | INCLUDE_CG={INCLUDE_CG} | excluded {len(EXCLUDE)} contingencies")
print(f"Eligible: |C_T|={len(CT_lines)} non-radial lines, |C_G|={len(CG_gens)} generators")
print(f"|C| = {len(set_C)} operating states (1 base + {len(set_C) - 1} contingencies)")

# Contingency configuration name (from the switches), and the output folder built as
# results/<formulation>/<config>/ so results never overwrite a different run.
config = {
    (False, False): "basecase",       # C = {0}
    (True,  True):  "contingencies",  # base + C_T + C_G
    (True,  False): "cont_trans",     # base + C_T (lines only)
    (False, True):  "cont_gen",       # base + C_G (generators only)
}[(INCLUDE_CT, INCLUDE_CG)]

OUT_DIR = f"results/{SOURCE}/{FORMULATION}/{config}"
print(f"MODEL={MODEL} | FORMULATION={FORMULATION} | config={config}")
print(f"output -> {OUT_DIR}")

INCLUDE_CT=False | INCLUDE_CG=False | excluded 3 contingencies
Eligible: |C_T|=175 non-radial lines, |C_G|=18 generators
|C| = 1 operating states (1 base + 0 contingencies)
MODEL=sc-dcopf | FORMULATION=redispatch_with_exclusions | config=basecase
output -> results/matpower/redispatch_with_exclusions/basecase


### 3.1 Scenario probabilities $\rho_c$

The base case carries most of the weight; the contingencies share the remainder equally.
The weights must sum to 1. (With only the base case, $\rho_0 = 1$.) These are parameters —
swap in failure-rate-based values later without touching the model structure.

In [18]:
# Per-element failure probabilities: every generator contingency has the same weight
# RHO_GEN, every line contingency the same RHO_LINE. Unlike splitting a fixed budget among
# scenarios, this makes each weight reflect the physical failure rate of that equipment type
# and does NOT depend on how many contingencies are in the set.
RHO_GEN  = 1e-2   # probability of a single generator failure
RHO_LINE = 1e-3   # probability of a single transmission (line) failure

par_rho = {}
for c in set_C:
    s = scen[c]
    if s["type"] == "gen":
        par_rho[c] = RHO_GEN
    elif s["type"] == "line":
        par_rho[c] = RHO_LINE
    # base case is filled below

# the base case takes the remaining probability: rho_0 = 1 - sum of all contingency rhos
rho_cont_sum = sum(par_rho.values())
par_rho[0] = 1.0 - rho_cont_sum

assert par_rho[0] > 0, (
    f"Contingency probabilities sum to {rho_cont_sum:.4f} (>1): lower RHO_GEN/RHO_LINE."
)
assert abs(sum(par_rho.values()) - 1.0) < 1e-9, "rho must sum to 1."
print(f"rho_gen={RHO_GEN}, rho_line={RHO_LINE}")
print(f"rho[0] (base) = {par_rho[0]:.4f}  (contingencies sum to {rho_cont_sum:.4f})")
print(f"sum of rho    = {sum(par_rho.values()):.6f}")

rho_gen=0.01, rho_line=0.001
rho[0] (base) = 1.0000  (contingencies sum to 0.0000)
sum of rho    = 1.000000


## 4. Model and decision variables

In [19]:
# create empty model
model = gp.Model("SC-DCOPF_118bus")
model.setParam("OutputFlag", 1)      # set to 1 to show the solver logs so I can debug the code
model.setParam("DualReductions", 0)  # so an infeasible model is reported as INFEASIBLE (not INF_OR_UNBD), enabling IIS

# create decision variables using addVar with free bounds to be limited by the constraints
var_p = {
    (g, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"p[{g},{c}]")
    for g in set_G for c in set_C # for each gen in set G AND each contingency in set c we create one P variable
}
var_f = {
    (l, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"f[{l},{c}]")
    for l in set_L for c in set_C # for each line in set L AND each contingency in set c we create one flow variable
}
var_theta = {
    (i, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"theta[{i},{c}]")
    for i in set_N for c in set_C # for each bus in set N AND each contingency in set c we create one theta variable
}

print(f"Created {len(var_p)} p, {len(var_f)} f, {len(var_theta)} theta variables.")

Set parameter OutputFlag to value 1
Set parameter DualReductions to value 0
Created 19 p, 186 f, 118 theta variables.


## 5. Objective

$$\min_{P,f,\theta} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

In [20]:
model.setObjective(
    gp.quicksum(par_rho[c] * par_cost[g] * var_p[(g, c)] * BASE_MVA # par_cost is in $/MWh and var_p is in p.u., so we multiply by BASE_MVA to get $/h
                for g in set_G for c in set_C),
    GRB.MINIMIZE,
)

## 6. Constraints

Each block below is one equation of the formulation, written for every scenario `c`.

### Energy Balance Constraints

$$\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{j \in \Omega_i^+} f_{ij,c} - \sum_{j \in \Omega_i^-} f_{ji,c}$$

In [21]:
for c in set_C:
    for i in set_N:
        gen_i = gp.quicksum(var_p[(g, c)] for g in set_gens_at[i])   # sum over G_i
        out_i = gp.quicksum(var_f[(l, c)] for l in set_out_lines[i]) # Omega_i^+
        in_i  = gp.quicksum(var_f[(l, c)] for l in set_in_lines[i])  # Omega_i^-
        model.addConstr(gen_i - par_pd[i] == out_i - in_i, name=f"bal[{i},{c}]")

### Generator Limits Constraints

$$P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c}$$

When a generator is out (`gen_on == 0`) both bounds collapse to 0, forcing $P_{g,c}=0$.

In [22]:
for c in set_C:
    for g in set_G:
        avail = gen_on(g, c)
        model.addConstr(var_p[(g, c)] >= par_pmin[g] * avail, name=f"pmin[{g},{c}]")
        model.addConstr(var_p[(g, c)] <= par_pmax[g] * avail, name=f"pmax[{g},{c}]")

### Thermal Transmission Limits

$$-F_{ij}^{\max} \le f_{ij,c} \le F_{ij}^{\max}$$

In [23]:
for c in set_C:
    for l in set_L:
        model.addConstr(var_f[(l, c)] >= -par_rate[l], name=f"fmin[{l},{c}]")
        model.addConstr(var_f[(l, c)] <=  par_rate[l], name=f"fmax[{l},{c}]")

### DC Branch Power Flow Equations

$$f_{ij,c} - \tfrac{1}{x_{ij}}(\theta_{i,c} - \theta_{j,c}) = 0 \;\; (N1_{ij,c}=1), \qquad f_{ij,c} = 0 \;\; (N1_{ij,c}=0)$$

If the line is in service we impose the DC flow law; if it is out, we force $f=0$ **without**
coupling the angles (avoids the 'ghost line' that would tie $\theta_i=\theta_j$).

In [24]:
for c in set_C:
    for l in set_L:
        i, j = par_from[l], par_to[l]
        if line_on(l, c) == 1:
            model.addConstr(
                var_f[(l, c)] == (1.0 / par_x[l]) * (var_theta[(i, c)] - var_theta[(j, c)]),
                name=f"dcflow[{l},{c}]",
            )
        else:
            model.addConstr(var_f[(l, c)] == 0.0, name=f"dcflow_off[{l},{c}]")

### Reference Bus Constraint

$$\theta_{\mathrm{ref},c} = 0$$

In [25]:
for c in set_C:
    model.addConstr(var_theta[(SLACK_BUS, c)] == 0.0, name=f"ref[{c}]")

print(f"Model built: {model.NumVars} variables, {model.NumConstrs} constraints.")

Model built: 0 variables, 0 constraints.


## 7. Solve

In [26]:
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
DualReductions  0

Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Optimize a model with 715 rows, 323 columns and 1360 nonzeros (Min)
Model fingerprint: 0x621d7045
Model has 19 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-02, 7e+01]

Presolve time: 0.03s
Presolved: 90 rows, 108 columns, 417 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      57    9.3152377e+04   0.000000e+00   0.000000e+00      0s

Solved in 57 iterations and 0.03 seconds (0.00 work units)
O

## 8. Results

With $\mathcal{C}=\{0\}$ this is a plain DCOPF, so total generation must equal total load and
the cost is the base-case dispatch cost — a sanity check before switching on contingencies.

In [27]:
if model.status == GRB.OPTIMAL:
    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    print("=" * 55)
    print("  SC-DCOPF SOLUTION — IEEE 118-Bus")
    print("=" * 55)
    print(f"  Expected cost   : ${model.ObjVal:,.2f} /h")
    print(f"  Total load      : {total_load:.2f} MW")
    print(f"  Total generation: {total_gen:.2f} MW (base case)")
    print(f"  Scenarios       : {len(set_C)}")
elif model.status == GRB.INFEASIBLE:
    import os, re
    print("INFEASIBLE. Computing IIS to locate the conflict...")
    model.computeIIS()

    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)

    # 1) full IIS file (Gurobi native) - the ground truth
    model.write(f"{OUT}/iis_full.ilp")

    # 2) deterministic per-scenario summary, parsed from constraint names like 'fmax[12,49]'
    #    (name pattern: <type>[..,<scenario_id>]). Only constraints flagged by the IIS are kept.
    pat = re.compile(r"^([a-zA-Z_]+)\[.*,(\d+)\]$")
    recs = []
    for con in model.getConstrs():
        if con.IISConstr:
            mm = pat.match(con.ConstrName)
            if mm:
                recs.append({"scenario_id": int(mm.group(2)), "ctype": mm.group(1)})
    iis_df = pd.DataFrame(recs)
    summary = (iis_df.groupby("scenario_id")
               .agg(n_constraints=("ctype", "size"),
                    constraint_types=("ctype", lambda s: ",".join(sorted(set(s)))))
               .reset_index())
    summary["scenario_desc"] = summary["scenario_id"].map(lambda c: scen[c]["desc"])

    # translate each conflicting scenario to the physical element that is out of service,
    # so 'scenario 49' is immediately readable as 'line 38-37' (or 'gen bus 10').
    par_label = dict(zip(branches["branch_id"], branches["label"]))
    def out_element(c):
        s = scen[c]
        if s["out_line"] is not None:
            return f"line {par_label[s['out_line']]} (branch_id {s['out_line']})"
        if s["out_gen"] is not None:
            return f"gen bus {par_g_bus[s['out_gen']]} (gen_id {s['out_gen']})"
        return "base case"
    summary["out_element"] = summary["scenario_id"].map(out_element)

    summary = summary[["scenario_id", "scenario_desc", "out_element",
                       "n_constraints", "constraint_types"]]
    summary.to_csv(f"{OUT}/iis_summary.csv", index=False)

    print(f"Saved iis_full.ilp and iis_summary.csv to {OUT}/")
    print(f"Conflicting scenarios: {summary['scenario_id'].tolist()}")
    display(summary)
else:
    print(f"Solver status: {model.status}")

  SC-DCOPF SOLUTION — IEEE 118-Bus
  Expected cost   : $93,152.38 /h
  Total load      : 4242.00 MW
  Total generation: 4242.00 MW (base case)
  Scenarios       : 1


In [28]:
# Base-case dispatch of the active generators
if model.status == GRB.OPTIMAL:
    rows = []
    for g in set_G:
        mw = var_p[(g, 0)].X * BASE_MVA
        if mw > 0.01:
            rows.append({"gen_id": g, "bus": par_g_bus[g],
                         "p_mw": round(mw, 2),
                         "pmax_mw": round(par_pmax[g] * BASE_MVA, 2),
                         "c1": round(par_cost[g], 3)})
    df_dispatch = pd.DataFrame(rows).sort_values("c1").reset_index(drop=True)
    print(f"Active generators: {len(df_dispatch)}")
    df_dispatch

Active generators: 13


## 9. Export results for documentation

Saves the base-case solution to `results/` (only when the model is optimal), so results can
be documented and reused. Three files:
- `basecase_summary.csv`     — cost, load, generation, scenario/generator counts
- `basecase_dispatch.csv`    — per-generator output and loading
- `basecase_line_loading.csv`— per-line flow and loading (sorted by loading), to spot congestion

In [29]:
import os
import math

if model.status == GRB.OPTIMAL:
    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)
    par_label = dict(zip(branches["branch_id"], branches["label"]))

    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    n_active   = sum(1 for g in set_G if var_p[(g, 0)].X * BASE_MVA > 0.01)

    # 1) summary
    pd.DataFrame({
        "metric": ["expected_cost_usd_per_h", "total_load_mw", "total_generation_mw",
                   "n_scenarios", "n_active_generators"],
        "value":  [round(model.ObjVal, 2), round(total_load, 2), round(total_gen, 2),
                   len(set_C), n_active],
    }).to_csv(f"{OUT}/summary.csv", index=False)

    # 2) per-generator dispatch (base case, c = 0)
    disp = []
    for g in set_G:
        mw   = var_p[(g, 0)].X * BASE_MVA
        pmax = par_pmax[g] * BASE_MVA
        disp.append({"gen_id": g, "bus": par_g_bus[g], "p_mw": round(mw, 2),
                     "pmax_mw": round(pmax, 2),
                     "loading_pct": round(100 * mw / pmax, 1) if pmax > 0 else 0,
                     "c1": round(par_cost[g], 3)})
    pd.DataFrame(disp).sort_values("c1").to_csv(f"{OUT}/dispatch.csv", index=False)

    # 3) per-line loading (base case) — sorted to reveal congestion
    fl = []
    for l in set_L:
        f    = var_f[(l, 0)].X * BASE_MVA
        fmax = par_rate[l] * BASE_MVA
        fl.append({"branch_id": l, "label": par_label[l], "f_mw": round(f, 2),
                   "fmax_mw": round(fmax, 2),
                   "loading_pct": round(100 * abs(f) / fmax, 1) if fmax > 0 else 0})
    (pd.DataFrame(fl).sort_values("loading_pct", ascending=False)
        .to_csv(f"{OUT}/line_loading.csv", index=False))

    # 4) bus voltage angles (base case). Slack is fixed at 0; others are relative to it.
    ang = []
    for i in set_N:
        rad = var_theta[(i, 0)].X
        ang.append({"bus": i, "theta_rad": round(rad, 6),
                    "theta_deg": round(math.degrees(rad), 4),
                    "is_slack": i == SLACK_BUS})
    (pd.DataFrame(ang).sort_values("theta_deg")
        .to_csv(f"{OUT}/angles.csv", index=False))

    print(f"Saved 4 files to {OUT}/ (cost=${model.ObjVal:,.2f}/h, {n_active} active gens).")
else:
    print("Model not optimal - nothing exported.")

Saved 4 files to results/matpower/redispatch_with_exclusions/basecase/ (cost=$93,152.38/h, 13 active gens).
